This notebook serves as a demonstration of the LASP SDTP application.

In [ ]:
# Some commands to make things easier to test within this notebook
from lasp_sdtp.database.database_controller import db
from tests import conftest
db.session.query(db.FileQueue).delete()
db.session.query(db.Transactions).delete()
db.session.query(db.Accounts).delete()
db.session.query(db.FileMetadata).delete()
db.session.commit()
db.insert_data('accounts', [{'username': 'ges_disc', 'role': 'subscriber', 'registration_open': True}])
conftest._add_file_metadata_entries()

# Configuration

In [ ]:
from lasp_sdtp.config import subscriber_config
subscriber_config

In [ ]:
subscriber_config['account_expiration_period'] = 1800
subscriber_config['checksum_type'] = 'sha256'
subscriber_config['distinguished_name'] = ''
subscriber_config['expiration_period'] = 180
subscriber_config['max_num_files'] = 10000
subscriber_config['num_download_threads'] = 5
subscriber_config['username'] = 'ges_disc'

In [ ]:
subscriber_config

# Register Account

Register a new account.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned.
    
Command: `curl PUT "http://127.0.0.1:8000/register -H "Cert-UID: <certificate>"`

In [ ]:
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

# List Available Files

Return a list of available files given user-supplied parameters.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with a list of the available files and their metadata.  If no parameters are supplied, then all available files are returned.

Command: `curl GET "http://127.0.0.1:8000/files?param1=value1&param2=value2... -H "Cert-UID: <certificate>"`

#### Supported parameters:
- `stream` (e.g. `prod`)
- `ShortName` (e.g. `TSIS_SC_L2`)
- `version` (e.g. `v01`)
- `date` (e.g.`2022-01-01`)
- `start_date` (e.g. `2022-01-01`)
- `end_date` (e.g. `2022-01-02`)


Note that `date` can only be supplied if `start_date` and/or `end_date` are *not* supplied.  Also, if `start_date` is supplied, then `end_date` *must* be supplied, or vice-versa 

### Examples

In [ ]:
# List all available files
! curl -i -X GET "http://127.0.0.1:8000/files" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# List all available files with ShortName of 'TSIS_SC_L2'
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# List all available files with specific ShortName and for specific date
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2&date=2022-01-02" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# List all available files between range between dates
! curl -i -X GET "http://127.0.0.1:8000/files?start_date=2021-12-31&end_date=2022-01-05" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

# Get Specific File

Return the contents of a file for a given `fileid`.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with the content of the file.  

Command: `curl GET "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [ ]:
! curl -i -X GET "http://127.0.0.1:8000/files/5981" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

# Delete Specific File

Remove a specific file from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [ ]:
! curl -i -X DELETE "http://127.0.0.1:8000/files/5981" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

# Delete Range of Files

Remove a range of files from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid_start>-<fileid_end> -H "Cert-UID: <certificate>"`

In [ ]:
# First get a few files so that they are in the queue
! curl -i -X GET "http://127.0.0.1:8000/files/5981" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/5982" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/5983" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# Delete the range of files
! curl -i -X DELETE "http://127.0.0.1:8000/files/5981-5983" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"